# F13 - CPU/GPU basics, random seeds, reproducibility, and experiment tracking

**Hardware:** CPU-only; optional GPU · **Time:** 3 hours · **Prerequisite:** F12

This notebook separates three ideas that beginners often merge: a random seed, a
deterministic algorithm, and a reproducible experiment record. It times a matrix
operation on the available device, trains a tiny model under three seeds, and
writes a machine-readable run log.

In [1]:
from pathlib import Path
import json
import platform
import random
import sys
import time

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
print("python", sys.version.split()[0], "| torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())
torch.set_num_threads(2)

python 3.12.10 | torch 2.4.1+cpu
cuda available: False


## 1. A seed controls pseudo-random sequences, not truth

`torch.manual_seed` resets PyTorch's generator. NumPy and Python's `random`
module have their own generators. Seed all three when a pipeline uses all three.
Two runs with the same seed on the same software and device should match; a
different device or library version may change the result.

In [2]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(123)
first = torch.randn(4)
set_seed(123)
second = torch.randn(4)
print("first:", first)
print("second:", second)
assert torch.equal(first, second)

first: tensor([-0.1115,  0.1204, -0.3696, -0.2404])
second: tensor([-0.1115,  0.1204, -0.3696, -0.2404])


## 2. CPU versus GPU timing

A GPU is fast for large parallel operations but has launch and transfer
overhead. Time after a warm-up, synchronize CUDA before stopping the clock, and
report the device name. A CPU result is valid; a GPU is an optimization, not a
requirement for this course.

In [3]:
size = 512
left = torch.randn(size, size)
right = torch.randn(size, size)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
left, right = left.to(device), right.to(device)

for _ in range(2):
    _ = left @ right
if device.type == "cuda":
    torch.cuda.synchronize()
start = time.perf_counter()
for _ in range(5):
    _ = left @ right
if device.type == "cuda":
    torch.cuda.synchronize()
elapsed = (time.perf_counter() - start) / 5

print("device:", device)
if device.type == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))
print(f"mean matmul time: {elapsed * 1000:.2f} ms")
assert elapsed > 0

device: cpu
mean matmul time: 0.95 ms


## 3. Determinism is a spectrum

`torch.use_deterministic_algorithms(True, warn_only=True)` asks PyTorch to use
deterministic kernels when available and warn otherwise. It can reduce speed and
does not make every GPU operation bitwise reproducible. Record the setting and
report variation instead of promising identical numbers across hardware.

In [4]:
torch.use_deterministic_algorithms(True, warn_only=True)
print("deterministic algorithms requested")
assert torch.are_deterministic_algorithms_enabled()

deterministic algorithms requested


## 4. A tiny experiment tracker

An experiment tracker does not need a server. A JSON Lines file with one record
per run, plus the code and data version, is enough to compare seeds. The logger
below appends a record and flushes it immediately so a crashed run still leaves
evidence.

In [5]:
class RunLogger:
    def __init__(self, path: Path):
        self.path = path

    def log(self, record: dict) -> None:
        with self.path.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(record, sort_keys=True) + "\n")

    def read(self) -> list[dict]:
        if not self.path.exists():
            return []
        return [json.loads(line) for line in self.path.read_text(encoding="utf-8").splitlines()]


log_path = OUTPUT_DIR / "f13_runs.jsonl"
log_path.unlink(missing_ok=True)
logger = RunLogger(log_path)
logger.log({"event": "start", "torch": torch.__version__, "device": str(device)})
assert len(logger.read()) == 1

## 5. Run the same model under three seeds

The model and data split are fixed; only the initialization and training noise
change. The spread across seeds is a lower bound on the uncertainty you should
report. One run is an anecdote.

In [6]:
X, y = make_moons(n_samples=500, noise=0.35, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train).astype("float32")
X_test = scaler.transform(X_test).astype("float32")
X_train_t = torch.tensor(X_train)
y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test_t = torch.tensor(X_test)
y_test_t = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

accuracies, validation_losses = [], []
for seed in [7, 17, 27]:
    set_seed(seed)
    model = nn.Sequential(nn.Linear(2, 12), nn.ReLU(), nn.Linear(12, 1))
    optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
    criterion = nn.BCEWithLogitsLoss()
    for _ in range(80):
        model.train()
        optimizer.zero_grad()
        loss = criterion(model(X_train_t), y_train_t)
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        predictions = (model(X_test_t) > 0).float()
        accuracy = (predictions == y_test_t).float().mean().item()
        validation_loss = criterion(model(X_test_t), y_test_t).item()
    accuracies.append(accuracy)
    validation_losses.append(validation_loss)
    logger.log(
        {
            "event": "run",
            "seed": seed,
            "model": "mlp_2_12_1",
            "optimizer": "adam",
            "lr": 0.05,
            "epochs": 80,
            "val_accuracy": round(accuracy, 4),
            "val_loss": round(validation_loss, 4),
            "device": str(device),
        }
    )
    print("seed", seed, "validation accuracy", round(accuracy, 3), "| loss", round(validation_loss, 3))

print("mean accuracy:", round(float(np.mean(accuracies)), 3))
print("std across seeds:", round(float(np.std(accuracies)), 3))
assert len(accuracies) == 3
assert all(0.5 < value <= 1.0 for value in accuracies)
assert float(np.std(accuracies)) > 0.005

seed 7 validation accuracy 0.848 | loss 0.314
seed 17 validation accuracy 0.872 | loss 0.257
seed 27 validation accuracy 0.904 | loss 0.246
mean accuracy: 0.875
std across seeds: 0.023


## 6. Read the log back and write an environment report

A run log is only useful if another person can read it. Parse the JSONL, verify
three runs, and write a compact report with versions and hardware. Do not include
usernames, absolute paths, tokens, or other personal data.

In [7]:
runs = [record for record in logger.read() if record.get("event") == "run"]
print(json.dumps(runs, indent=2))
assert len(runs) == 3

report = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "numpy": np.__version__,
    "platform": platform.platform(),
    "device": str(device),
    "device_name": torch.cuda.get_device_name(0) if device.type == "cuda" else platform.processor(),
    "deterministic_requested": True,
    "seeds": [record["seed"] for record in runs],
    "mean_val_accuracy": round(float(np.mean([record["val_accuracy"] for record in runs])), 4),
    "std_val_accuracy": round(float(np.std([record["val_accuracy"] for record in runs])), 4),
}
report_path = OUTPUT_DIR / "f13_environment_report.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print("wrote", report_path)
assert report_path.exists()

[
  {
    "device": "cpu",
    "epochs": 80,
    "event": "run",
    "lr": 0.05,
    "model": "mlp_2_12_1",
    "optimizer": "adam",
    "seed": 7,
    "val_accuracy": 0.848,
    "val_loss": 0.3144
  },
  {
    "device": "cpu",
    "epochs": 80,
    "event": "run",
    "lr": 0.05,
    "model": "mlp_2_12_1",
    "optimizer": "adam",
    "seed": 17,
    "val_accuracy": 0.872,
    "val_loss": 0.2572
  },
  {
    "device": "cpu",
    "epochs": 80,
    "event": "run",
    "lr": 0.05,
    "model": "mlp_2_12_1",
    "optimizer": "adam",
    "seed": 27,
    "val_accuracy": 0.904,
    "val_loss": 0.2456
  }
]
wrote outputs\f13_environment_report.json


## Check your work

- [ ] You seeded Python, NumPy, and PyTorch.
- [ ] You reported the device used for timing.
- [ ] You can explain why determinism is not guaranteed across hardware.
- [ ] Your JSONL log has one record per seed.
- [ ] `outputs/f13_environment_report.json` exists and contains no personal data.

This completes the foundation sequence. Continue to the approved M02/M03 module
plans or the next level in the syllabus.